In [ ]:
!pip install evaluate
import numpy as np
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer
from transformers import DataCollatorWithPadding
import evaluate
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

SEED = 2026

# Load data
train_emo_label = pd.read_csv("files/emotion.csv")
texts = pd.read_json("files/final_posts.json")
data_id = pd.read_csv("files/data_identification.csv")

# Combine data and slice into train, test
df = pd.merge(texts, data_id, on="post_id", how="outer")
train_df = pd.merge(df, train_emo_label, on="post_id", how="inner")
test_df = df[df["set"] == "test"].reset_index(drop=True)

# Overview
print("Training data:", train_df.shape)
print("Test data:", test_df.shape)

Training data: (3360, 4)
Test data: (1440, 3)


In [ ]:
def clean_text(s: str) -> str:
    s = s.lower()
    s = re.sub(r"http\S+|www\.\S+", " ", s)
    s = re.sub(r"@\w+", " ", s)
    return re.sub(r"\s+", " ", s).strip()

# Create a copy of cleaned text in "clean"
train_df["clean"] = train_df["text"].map(clean_text)
test_df["clean"] = test_df["text"].map(clean_text)

for i in range(0,3):
    print("Train (raw):", train_df["text"].iloc[i])
    print("Train (cleaned):", train_df["clean"].iloc[i])
    print()
for i in range(0,3):
    print("Test (raw):", test_df["text"].iloc[i])
    print("Test (cleaned):", test_df["clean"].iloc[i])
    print()

Train (raw): >just because I miss [NAME] and [NAME], who played them. You forgot to add the rest of that sentence 
Train (cleaned): >just because i miss [name] and [name], who played them. you forgot to add the rest of that sentence

Train (raw): Even one second gets you killed against focus fire, especially if you're discorded, which is almost always in this meta. 
Train (cleaned): even one second gets you killed against focus fire, especially if you're discorded, which is almost always in this meta.

Train (raw): Stargazing with Vi is definitely my favorite moment of the season so far
Train (cleaned): stargazing with vi is definitely my favorite moment of the season so far

Test (raw): I feel happy and satisfied today when I eat delicious food.
Test (cleaned): i feel happy and satisfied today when i eat delicious food.

Test (raw): I'm in a really bad mood, I don't want to talk to anyone, so shut it down and #movietime.
Test (cleaned): i'm in a really bad mood, i don't want to talk t

In [ ]:
#
emotions = sorted(train_df["label"].unique())
print("Emotions:", emotions)
label2id = {lab: id for id, lab in enumerate(emotions)}
id2label = {id: lab for lab, id in label2id.items()}
train_df["y_id"] = train_df["label"].map(label2id)

# Split train data (20% for test)
splt_train_df, splt_test_df = train_test_split(train_df, test_size=0.2, stratify=train_df["y_id"], random_state=SEED)
print("Split training data:", splt_train_df.shape)
print("Split test data:", splt_test_df.shape)

Emotions: ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']
Split training data: (2688, 6)
Split test data: (672, 6)


In [ ]:

emotion_ds = DatasetDict({
    "train":      Dataset.from_pandas(splt_train_df[["clean", "y_id"]].rename(columns={"clean": "text", "y_id": "label"}), preserve_index=False),
    "validation": Dataset.from_pandas(splt_test_df[["clean", "y_id"]].rename(columns={"clean": "text", "y_id": "label"}), preserve_index=False),
    "test":       Dataset.from_pandas(test_df[["clean"]].rename(columns={"clean": "text"}), preserve_index=False),
})

print(emotion_ds)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 2688
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 672
    })
    test: Dataset({
        features: ['text'],
        num_rows: 1440
    })
})


In [ ]:


tokenizer = AutoTokenizer.from_pretrained("roberta-base")

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

tokenized_emotion = emotion_ds.map(preprocess_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/2688 [00:00<?, ? examples/s]

Map:   0%|          | 0/672 [00:00<?, ? examples/s]

Map:   0%|          | 0/1440 [00:00<?, ? examples/s]

In [ ]:

f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return f1.compute(predictions=predictions, references=labels, average="macro")

In [ ]:

model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base", num_labels=len(emotions), id2label=id2label, label2id=label2id
)

training_args = TrainingArguments(
    output_dir="emotion_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=6,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,
    seed=SEED,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_emotion["train"],
    eval_dataset=tokenized_emotion["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1
1,No log,1.244962,0.435008
2,No log,1.112328,0.590951
3,1.170424,1.130778,0.607316
4,1.170424,1.194375,0.608642
5,1.170424,1.295007,0.602345
6,0.528721,1.311585,0.610397


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1008, training_loss=0.8461933334668478, metrics={'train_runtime': 168.7962, 'train_samples_per_second': 95.547, 'train_steps_per_second': 5.972, 'total_flos': 516702891363840.0, 'train_loss': 0.8461933334668478, 'epoch': 6.0})

In [ ]:
test_pred = trainer.predict(tokenized_emotion["test"])
pred_emotion_ids = np.argmax(test_pred.predictions, axis=1)
test_df["label"] = [model.config.id2label[int(i)] for i in pred_emotion_ids]

In [ ]:
# Output result to submission.csv
sample_sub = pd.read_csv("files/sampleSubmission.csv")
result_sub = pd.merge(sample_sub[["post_id"]], test_df[["post_id", "label"]], on="post_id", how="left")
result_sub = result_sub.rename(columns={"post_id": "id"})
result_sub.to_csv("submission.csv", index=False)

# Check
sub = pd.read_csv("submission.csv")
print(sub.shape)
sub.head()

(1440, 2)


,id,label
0,xyzx9jp,surprise
1,l3vzzxt,sadness
2,g680iug,sadness
3,m4t8bp6,joy
4,ai3al6e,surprise
